Dashboard Prep.

In [11]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
from pathlib import Path
import pandas as pd
import numpy as np
import json, shutil
import warnings
warnings.filterwarnings('ignore')

PROJECT_ID = 'cloudiq-analytics'
DATASET    = 'bigquery-public-data.thelook_ecommerce'
MY_DATASET = f'{PROJECT_ID}.cloudiq'

client    = bigquery.Client(project=PROJECT_ID)
FIGURES   = Path('/content/figures')
PROCESSED = Path('/content/processed')
FIGURES.mkdir(exist_ok=True)
PROCESSED.mkdir(exist_ok=True)

def run_query(sql):
    return client.query(sql).to_dataframe()

print("Ready.")


Ready.


Re-upload processed files from earlier notebooks

In [12]:
# If you are running this in a fresh Colab session, the /content/processed/
# files from notebooks 01–03 will be gone. Upload the zip you downloaded.

from google.colab import files

print("Upload your previously downloaded outputs zip (or individual CSVs).")
print("If you are continuing in the SAME session, skip this cell.")

# Uncomment if needed:
# uploaded = files.upload()
# import zipfile
# with zipfile.ZipFile(list(uploaded.keys())[0]) as z:
#     z.extractall('/content/')
# print("Files restored.")

# Check what we have
existing = list(PROCESSED.glob('*.csv'))
print(f"\nFound {len(existing)} CSV files in /content/processed/:")
for f in sorted(existing):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<45}  {size_kb:6.1f} KB")


Upload your previously downloaded outputs zip (or individual CSVs).
If you are continuing in the SAME session, skip this cell.

Found 18 CSV files in /content/processed/:
  bqml_evaluation.csv                               0.2 KB
  bqml_feature_weights.csv                          0.9 KB
  cohort_retention.csv                            146.7 KB
  conversion_funnel.csv                             0.2 KB
  funnel_by_source.csv                              0.3 KB
  kpis_summary.csv                                  0.2 KB
  monthly_growth.csv                                6.4 KB
  monthly_revenue.csv                               4.6 KB
  new_vs_returning.csv                              3.1 KB
  order_status.csv                                  0.2 KB
  repeat_purchase_predictions.csv                  27.8 KB
  return_rate_by_category.csv                       0.7 KB
  revenue_by_category.csv                           1.1 KB
  revenue_by_country.csv                            0.6 KB
  r

In [13]:
# Pull top-level KPIs fresh from BigQuery for the dashboard header cards
kpis_sql = f"""
WITH order_metrics AS (
    SELECT
        COUNT(DISTINCT o.order_id)                           AS total_orders,
        COUNT(DISTINCT o.user_id)                            AS total_customers,
        ROUND(SUM(oi.sale_price), 2)                         AS total_revenue,
        ROUND(AVG(oi.sale_price), 2)                         AS avg_order_value,
        ROUND(COUNTIF(o.status = 'Returned') /
              COUNT(*) * 100, 2)                             AS return_rate_pct,
        MIN(DATE(o.created_at))                              AS data_start,
        MAX(DATE(o.created_at))                              AS data_end
    FROM `{DATASET}.orders` o
    JOIN `{DATASET}.order_items` oi ON o.order_id = oi.order_id
    WHERE o.status NOT IN ('Cancelled')
),
repeat_metrics AS (
    SELECT
        COUNT(DISTINCT user_id)                              AS total_scored,
        COUNTIF(repeat_buyer = 1)                            AS actual_repeats,
        ROUND(COUNTIF(repeat_buyer = 1) /
              COUNT(*) * 100, 2)                             AS repeat_rate_pct
    FROM `{MY_DATASET}.repeat_purchase_training`
)
SELECT om.*, rm.repeat_rate_pct
FROM order_metrics om, repeat_metrics rm
"""
kpis_df = run_query(kpis_sql)
kpis_df.to_csv(PROCESSED / 'kpis_summary.csv', index=False)

print("Master KPIs:")
for col in kpis_df.columns:
    print(f"  {col:<30}: {kpis_df[col].values[0]}")


Master KPIs:
  total_orders                  : 105818
  total_customers               : 71786
  total_revenue                 : 9140239.04
  avg_order_value               : 59.48
  return_rate_pct               : 11.74
  data_start                    : 2019-01-14
  data_end                      : 2026-10-29
  repeat_rate_pct               : 22.66


In [14]:
# Top 500 highest repeat-probability customers for dashboard table
predict_sql = f"""
SELECT
    p.user_id,
    ROUND(p.predicted_repeat_buyer_probs[OFFSET(1)].prob, 4) AS repeat_prob,
    p.predicted_repeat_buyer                                  AS predicted_label,
    t.age,
    t.gender,
    t.country,
    t.first_order_value,
    t.first_category,
    t.traffic_source,
    t.repeat_buyer                                            AS actual_label
FROM ML.PREDICT(
    MODEL `{MY_DATASET}.repeat_buyer_model`,
    (SELECT * FROM `{MY_DATASET}.repeat_purchase_training`)
) p
JOIN `{MY_DATASET}.repeat_purchase_training` t ON p.user_id = t.user_id
ORDER BY repeat_prob DESC
LIMIT 500
"""
predictions_df = run_query(predict_sql)
predictions_df.to_csv(PROCESSED / 'repeat_purchase_predictions.csv', index=False)
print(f"Saved top-500 predictions: {len(predictions_df)} rows")
print(predictions_df.head(5).to_string(index=False))


Saved top-500 predictions: 500 rows
 user_id  repeat_prob  predicted_label  age gender       country  first_order_value first_category traffic_source  actual_label
   97126       0.9949                0   51      M     Australia              14.99    Accessories        Adwords             0
   13783       0.9945                0   60      M United States              25.00    Accessories        Adwords             0
    5798       0.9941                0   51      M         China               9.99    Accessories        Adwords             0
   87277       0.9935                0   52      M United States              23.99         Active        Adwords             0
   76220       0.9935                0   27      M         China               9.95    Accessories        Adwords             0


In [15]:
required_files = [
    'kpis_summary.csv',
    'monthly_revenue.csv',
    'monthly_growth.csv',
    'revenue_by_category.csv',
    'revenue_by_country.csv',
    'top_products.csv',
    'order_status.csv',
    'return_rate_by_category.csv',
    'cohort_retention.csv',
    'new_vs_returning.csv',
    'conversion_funnel.csv',
    'funnel_by_source.csv',
    'time_to_purchase.csv',
    'rfm_scores.csv',
    'rfm_segment_summary.csv',
    'bqml_evaluation.csv',
    'bqml_feature_weights.csv',
    'repeat_purchase_predictions.csv',
]

print("File check:")
all_ok = True
for fname in required_files:
    path = PROCESSED / fname
    if path.exists():
        size_kb = path.stat().st_size / 1024
        print(f"  ✓  {fname:<50}  {size_kb:6.1f} KB")
    else:
        print(f"  ✗  {fname:<50}  MISSING — re-run the relevant notebook")
        all_ok = False

print()
if all_ok:
    print("All files present. Ready to build the Streamlit dashboard.")
else:
    print("Some files missing. Re-run the notebook that produces them.")


File check:
  ✓  kpis_summary.csv                                       0.2 KB
  ✓  monthly_revenue.csv                                    4.6 KB
  ✓  monthly_growth.csv                                     6.4 KB
  ✓  revenue_by_category.csv                                1.1 KB
  ✓  revenue_by_country.csv                                 0.6 KB
  ✓  top_products.csv                                       1.4 KB
  ✓  order_status.csv                                       0.2 KB
  ✓  return_rate_by_category.csv                            0.7 KB
  ✓  cohort_retention.csv                                 146.7 KB
  ✓  new_vs_returning.csv                                   3.1 KB
  ✓  conversion_funnel.csv                                  0.2 KB
  ✓  funnel_by_source.csv                                   0.3 KB
  ✓  time_to_purchase.csv                                   0.5 KB
  ✓  rfm_scores.csv                                      3082.0 KB
  ✓  rfm_segment_summary.csv                      